## Setup & Imports

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('.'))

import pandas as pd
import numpy as np
from pathlib import Path
import json
from datetime import datetime

# Import project modules
from src.deid import deidentify_text, deidentify_csv
from src.data_ingest import load_csv, clean_text, preprocess_dataframe
from src.generate_scenarios import generate_scenarios
from src.risk_assessment import assess_risk
from src.explain import explain_text
from src.evaluate import evaluate
from src.plots import generate_plots

print("✓ Imports successful")

## 1. Generate Synthetic Clinical Scenarios

In [ ]:
# Generate 20 synthetic scenarios for demonstration
scenarios = generate_scenarios(n=20, seed=42)

print(f"Generated {len(scenarios)} scenarios")
print(f"\nRisk distribution:")
print(scenarios['label'].value_counts())
print(f"\nFirst 3 scenarios:")
scenarios.head(3)

## 2. De-identification (PHI Removal)

In [ ]:
# Example: De-identify a clinical note
sample_text = "Patient John Smith (SSN 123-45-6789) called at 555-1234 on 2023-01-15. Age 65. MR#ABC123. Email: john@example.com"

print("Original:")
print(sample_text)
print("\nDe-identified:")
deidentified = deidentify_text(sample_text)
print(deidentified)

## 3. Risk Assessment

Classify clinical scenarios into risk levels: low, medium, or high

In [ ]:
# Assess risk for first 5 scenarios
results = []

for idx, row in scenarios.head(5).iterrows():
    text = row['text']
    risk = assess_risk(text)
    
    results.append({
        'scenario': idx,
        'true_label': row['label'],
        'predicted_risk': risk['risk_level'],
        'score': risk['score'],
        'model_based': risk['model_based']
    })
    
    print(f"\nScenario {idx}:")
    print(f"  Text: {text[:80]}...")
    print(f"  True: {row['label']} | Predicted: {risk['risk_level']} (score: {risk['score']:.2f})")

results_df = pd.DataFrame(results)
print(f"\n\nResults table:")
results_df

## 4. Feature-Level Explanations

Understand which features (words/phrases) contribute to each prediction

In [ ]:
# Get explanations for a high-risk scenario
high_risk_text = scenarios[scenarios['label'] == 'high'].iloc[0]['text']

print("Text:")
print(high_risk_text)
print("\n" + "="*50)

explanations = explain_text(high_risk_text)

print(f"\nTop contributing features:")
if explanations['features']:
    for i, (feature, weight) in enumerate(explanations['features'][:5], 1):
        print(f"  {i}. '{feature}' (weight: {weight:.3f})")
else:
    print("  (No explanations available - model not yet trained)")

## 5. End-to-End Evaluation Pipeline

In [ ]:
# Run comprehensive evaluation on 50 scenarios
print("Running evaluation pipeline...")
print("This may take 30-60 seconds.\n")

eval_result = evaluate(n=50, output_dir='reports/demo_eval')

print(f"✓ Evaluation complete!")
print(f"\nResults saved to: {eval_result['out_dir']}")
print(f"\nMetrics:")

# Load and display metrics
metrics_file = Path(eval_result['out_dir']) / 'metrics.json'
if metrics_file.exists():
    with open(metrics_file) as f:
        metrics = json.load(f)
    
    print(f"  Overall Accuracy: {metrics.get('accuracy', 'N/A')}")
    print(f"  Per-class Recall: {metrics.get('per_class_recall', 'N/A')}")
    print(f"  Per-class Precision: {metrics.get('per_class_precision', 'N/A')}")

## 6. Load and Analyze Results

In [ ]:
# Load evaluation results
results_csv = Path(eval_result['results_csv'])

if results_csv.exists():
    eval_df = pd.read_csv(results_csv)
    
    print(f"Evaluation results shape: {eval_df.shape}")
    print(f"\nColumns: {list(eval_df.columns)}")
    print(f"\nFirst 5 predictions:")
    eval_df[['text', 'true_label', 'predicted_label', 'confidence']].head()
else:
    print("Results CSV not found (models may not be trained yet)")

## 7. Performance Analysis

In [ ]:
# Analyze prediction distribution
if results_csv.exists():
    print("Prediction distribution:")
    print(eval_df['predicted_label'].value_counts())
    
    print("\nTrue label distribution:")
    print(eval_df['true_label'].value_counts())
    
    print("\nConfidence statistics:")
    print(eval_df['confidence'].describe())
    
    # Misclassifications
    misclassified = eval_df[eval_df['true_label'] != eval_df['predicted_label']]
    print(f"\nMisclassification rate: {len(misclassified)/len(eval_df):.2%}")
    print(f"\nMisclassification examples:")
    misclassified[['text', 'true_label', 'predicted_label']].head(3)

## 8. Visualization

Generate evaluation plots

In [ ]:
# Generate plots from evaluation results
if results_csv.exists():
    print("Generating plots...")
    try:
        generate_plots(str(results_csv))
        print("✓ Plots generated successfully!")
        print("\nPlots saved to: docs/figures/")
    except Exception as e:
        print(f"Plotting skipped: {e}")
else:
    print("Cannot generate plots (results not available)")

## 9. Summary & Next Steps

In [ ]:
print("""
╔═══════════════════════════════════════════════════════════════╗
║           PATIENT SAFETY LLM PIPELINE SUMMARY                   ║
╠═══════════════════════════════════════════════════════════════╣
║                                                               ║
║  ✓ Data ingestion and preprocessing                          ║
║  ✓ De-identification of protected health information         ║
║  ✓ Risk assessment and classification                        ║
║  ✓ Feature-level explanations                                ║
║  ✓ End-to-end evaluation pipeline                            ║
║  ✓ Result visualization and analysis                         ║
║                                                               ║
╠═══════════════════════════════════════════════════════════════╣
║                        NEXT STEPS                              ║
╠═══════════════════════════════════════════════════════════════╣
║                                                               ║
║  1. Run baseline evaluation with full dataset (200 scenarios) ║
║     → See: reports/eval_*/evaluation_results.csv             ║
║                                                               ║
║  2. Train cross-validated models with augmentation            ║
║     → See: reports/cv_report_*/cv_metrics.json              ║
║                                                               ║
║  3. Review manuscript and results                            ║
║     → See: docs/study_draft.md                              ║
║                                                               ║
║  4. Deploy API or Streamlit UI                               ║
║     → API: uvicorn src.app:app --reload                     ║
║     → UI: streamlit run src/ui.py                           ║
║                                                               ║
║  5. Validate on real clinical data (future work)             ║
║     → See: DEPLOYMENT_GUIDE.md                              ║
║                                                               ║
╚═══════════════════════════════════════════════════════════════╝

For more information, see:
  • README.md — Project overview
  • docs/study_draft.md — Full manuscript
  • SUPPLEMENTARY_MATERIALS.md — Data specs and reproducibility
  • DEPLOYMENT_GUIDE.md — Production deployment
""")